# 02 — Joint Transformer Grade Prediction

## From Language Modeling to Grade Prediction

In NLP, **BERT-style models** are encoder-only transformers that take a sequence of tokens, process them through multiple self-attention layers, and produce a single output (like a classification label). The key insight is:

- **Input**: A sequence of tokens (words, subwords, or in our case, holds)
- **Processing**: Multiple layers of self-attention that let each token "look at" every other token
- **Output**: A pooled representation (typically from a `[CLS]` token) that summarizes the entire sequence

### Our architecture

We use a **Transformer Encoder** (similar to BERT) with these components:

1. **Token embeddings**: Convert integer token IDs to dense vectors
2. **Positional embeddings**: Tell the model where each token is in the sequence
3. **Coordinate features**: Inject physical (x, y) position of each hold into the embedding
4. **Transformer encoder layers**: Multiple layers of self-attention + feedforward
5. **Regression head**: Take the `<CLS>` token's output and predict a single difficulty score

### Why this works for climbing

A climb's difficulty depends on the *relationships between holds*, not just individual holds. Self-attention naturally captures these relationships:

- A start hold far from the first middle hold suggests a big opening move
- Two hand holds close together with a foot hold far away suggests a dyno
- The overall spatial distribution determines the "flow" of the climb

The transformer can learn these spatial relationships through attention, without us having to manually engineer features like "mean hand reach" or "height gained" (though those features were useful in the classical model).

### Input format

```text
<CLS> <BOARD_TB2> <ANGLE_40> <TB2_p344_start> <TB2_p369_middle> ... <TB2_p603_finish>
```

Note: We use `<CLS>` instead of `<BOS>` and we **exclude the grade token** — the model must predict the grade, not see it!

### Target

```text
display_difficulty (continuous value, e.g., 20.5)
```

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from climbingboardgpt.datasets import RouteGradeDataset
from climbingboardgpt.metrics import regression_metrics, metrics_by_board
from climbingboardgpt.models import JointRouteTransformerRegressor

In [ ]:
TOKENIZED = ROOT / "data" / "processed" / "tokenized"
df_routes = pd.read_csv(TOKENIZED / "route_sequences.csv")
vocab = json.loads((TOKENIZED / "token_vocab.json").read_text(encoding="utf-8"))

stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = {int(k): str(v) for k, v in vocab["itos"].items()}
df_token_meta = pd.read_csv(TOKENIZED / "token_metadata.csv")

pad_id = stoi["<PAD>"]
unk_id = stoi["<UNK>"]

print(f"Vocabulary size: {len(stoi):,}")
print(f"Total routes: {len(df_routes):,}")

## Build model IDs and coordinate features

### Coordinate features: Why inject physical position?

In standard NLP, positional embeddings tell the model *which position in the sequence* a token occupies. But for climbing, the **physical position on the wall** matters more than the sequence position.

We create a 3-dimensional feature vector for each token:
1. `x_norm`: Normalized horizontal position on the board (-1 to 1)
2. `y_norm`: Normalized vertical position on the board (-1 to 1)
3. `is_hold`: 1 if this token represents a hold, 0 otherwise

These features are projected through a linear layer and added to the token embeddings. This is similar to how some vision-language models inject spatial features from images alongside text tokens.

In [ ]:
def encode(tokens):
    """Convert a list of token strings to integer IDs."""
    return [stoi.get(token, unk_id) for token in tokens]

# Prepare input sequences for the grade predictor
# We use the "no grade" version because the model should predict the grade,
# not see it in the input!
# We also prepend <CLS> which will be used for pooling the sequence representation
df_routes["tokens_no_grade"] = df_routes["sequence_no_grade"].fillna("").str.split()
df_routes["model_tokens"] = df_routes["tokens_no_grade"].apply(
    lambda tokens: ["<CLS>"] + tokens[1:] if tokens else ["<CLS>"]
)
df_routes["model_ids"] = df_routes["model_tokens"].apply(encode)
df_routes["seq_len"] = df_routes["model_ids"].apply(len)
max_len = int(df_routes["seq_len"].max())

# Build coordinate features matrix: (vocab_size, 3)
# Each row corresponds to a token ID and contains [x_norm, y_norm, is_hold]
# This will be used as additional input to the model alongside token embeddings
coord_features = np.zeros((len(stoi), 3), dtype=np.float32)
for _, row in df_token_meta.iterrows():
    token_id = int(row["token_id"])
    coord_features[token_id, 0] = 0.0 if pd.isna(row.get("x_norm", 0.0)) else float(row.get("x_norm", 0.0))
    coord_features[token_id, 1] = 0.0 if pd.isna(row.get("y_norm", 0.0)) else float(row.get("y_norm", 0.0))
    coord_features[token_id, 2] = 0.0 if pd.isna(row.get("is_hold", 0.0)) else float(row.get("is_hold", 0.0))
coord_features = torch.tensor(coord_features, dtype=torch.float32)

print(f"Max sequence length: {max_len}")
print(f"Coordinate features shape: {coord_features.shape}")
print(f"Vocabulary size: {len(stoi)}")

## Data loaders

### Batching and padding

Transformers process data in **batches** for efficiency. But routes have different lengths (different numbers of holds). We handle this by:

1. **Padding**: Shorter sequences are padded with `<PAD>` tokens to match the longest sequence in the batch
2. **Attention masking**: The model receives a binary mask that tells it which positions are real data vs padding

This is exactly how BERT and GPT handle variable-length text sequences.

### The RouteGradeDataset class

For each route, this dataset produces:
- `input_ids`: Integer token IDs, padded to `max_len`
- `attention_mask`: 1 for real tokens, 0 for padding
- `target`: The difficulty score we want to predict
- `uuid`, `board_key`: Metadata for evaluation

In [ ]:
train_df = df_routes[df_routes["split"] == "train"].reset_index(drop=True)
val_df = df_routes[df_routes["split"] == "val"].reset_index(drop=True)
test_df = df_routes[df_routes["split"] == "test"].reset_index(drop=True)

train_ds = RouteGradeDataset(train_df, max_len=max_len, pad_id=pad_id)
val_ds = RouteGradeDataset(val_df, max_len=max_len, pad_id=pad_id)
test_ds = RouteGradeDataset(test_df, max_len=max_len, pad_id=pad_id)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print(f"Training samples: {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")
print(f"Test samples: {len(test_ds):,}")

## Model Architecture

### The JointRouteTransformerRegressor

This model is a **transformer encoder** with a regression head. Here's what each component does:

1. **Token embedding** (`nn.Embedding`): Converts integer token IDs to dense vectors of dimension `d_model`. This is the same as word embeddings in NLP.

2. **Positional embedding** (`nn.Embedding`): Adds position information so the model knows which position each token occupies. Unlike sinusoidal positional encodings in the original Transformer paper, we use learned embeddings.

3. **Coordinate projection** (`nn.Linear`): Projects the 3-dimensional coordinate features (x_norm, y_norm, is_hold) to `d_model` dimensions and adds them to the token embeddings. This injects physical position information.

4. **Transformer encoder** (`nn.TransformerEncoder`): Multiple layers of self-attention and feedforward networks. Each layer:
   - Computes self-attention: every hold "looks at" every other hold
   - Applies feedforward transformation
   - Uses residual connections and layer normalization

5. **Regression head**: Takes the `<CLS>` token's output (which has aggregated information from the entire sequence) and predicts a single difficulty score.

### Hyperparameters

- `d_model=128`: The dimensionality of embeddings and hidden states
- `nhead=4`: Number of attention heads (multi-head attention)
- `num_layers=4`: Number of transformer layers
- `dim_feedforward=256`: Dimension of the feedforward network inside each layer
- `dropout=0.10`: Dropout probability for regularization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JointRouteTransformerRegressor(
    vocab_size=len(stoi),
    max_len=max_len,
    coord_features=coord_features,
    d_model=128,
    nhead=4,
    num_layers=4,
    dim_feedforward=256,
    dropout=0.10,
    pad_id=pad_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Configuration

### Loss function: MSE (Mean Squared Error)

We use MSE loss because we're predicting a continuous value (difficulty score). This penalizes large errors more than small ones, which is appropriate for grade prediction.

### Optimizer: AdamW

AdamW is the standard optimizer for transformer models. It combines:
- **Adam**: Adaptive learning rates per parameter
- **Weight decay**: L2 regularization to prevent overfitting

### Early stopping

We stop training if validation loss doesn't improve for `patience` epochs. This prevents overfitting and saves compute.

In [ ]:
def run_epoch(model, loader, device, optimizer=None):
    """Run one epoch of training or evaluation.
    
    The RouteGradeDataset returns a dictionary with keys:
    - input_ids: token IDs, shape (batch_size, seq_len)
    - attention_mask: binary mask, shape (batch_size, seq_len)
    - target: difficulty score, shape (batch_size,)
    - uuid: route identifiers (for logging)
    - board_key: board identifiers (for logging)
    """
    is_train = optimizer is not None
    model.train(is_train)
    criterion = nn.MSELoss()

    losses, preds, targets, uuids, boards = [], [], [], [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        target = batch["target"].to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        pred = model(input_ids, attention_mask)
        loss = criterion(pred, target)

        if is_train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        losses.append(loss.item() * input_ids.size(0))
        preds.extend(pred.detach().cpu().numpy().tolist())
        targets.extend(target.detach().cpu().numpy().tolist())
        uuids.extend(batch["uuid"])
        boards.extend(batch["board_key"])

    avg_loss = sum(losses) / max(1, len(loader.dataset))
    return avg_loss, np.asarray(preds), np.asarray(targets), uuids, boards


# Training configuration
num_epochs = 30
patience = 12

print(f"Max epochs: {num_epochs}")
print(f"Early stopping patience: {patience}")

## Training Loop

The training loop follows the standard deep learning workflow:

1. **Forward pass**: Feed input through the model to get predictions
2. **Compute loss**: Compare predictions to actual grades using MSE
3. **Backward pass**: Compute gradients via backpropagation
4. **Update weights**: Adjust model parameters using the optimizer
5. **Validate**: Check performance on held-out validation data
6. **Early stopping**: Stop if validation loss stops improving

We track both fine-grained metrics (MAE, RMSE) and practical metrics (V-grade accuracy within ±1 grade).

In [ ]:
history = []
best_val_mae = float("inf")
best_state = None
best_epoch = 0
epochs_without_improvement = 0

print("Starting training...\n")

for epoch in range(1, num_epochs + 1):
    train_loss, train_pred, train_true, _, _ = run_epoch(model, train_loader, device, optimizer)
    val_loss, val_pred, val_true, _, _ = run_epoch(model, val_loader, device, optimizer=None)

    train_metrics = regression_metrics(train_true, train_pred)
    val_metrics = regression_metrics(val_true, val_pred)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_mae": train_metrics["mae"],
        "val_mae": val_metrics["mae"],
        "train_r2": train_metrics["r2"],
        "val_r2": val_metrics["r2"],
        "val_within_1_vgrade": val_metrics["within_1_vgrade"],
    })

    if val_metrics["mae"] < best_val_mae:
        best_val_mae = val_metrics["mae"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0 or epoch == best_epoch:
        print(
            f"Epoch {epoch:03d} | "
            f"train MAE {train_metrics['mae']:.3f} | "
            f"val MAE {val_metrics['mae']:.3f} | "
            f"val R² {val_metrics['r2']:.3f} | "
            f"val ±1V {val_metrics['within_1_vgrade']:.1f}%"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nTraining complete. Best epoch: {best_epoch}, Best val MAE: {best_val_mae:.4f}")

## Test Set Evaluation

After training, we load the best model (based on validation MAE) and evaluate on the held-out test set. We report:

- **MAE** (Mean Absolute Error): Average error in difficulty score points
- **RMSE** (Root Mean Squared Error): Penalizes large errors more
- **R²** (R-squared): How much variance in grades the model explains
- **Within ±1 difficulty**: Percentage of predictions within 1 point
- **Within ±1 V-grade**: Percentage of predictions within 1 V-grade

We also break down performance by board (TB2 vs Kilter) to check for bias.

In [ ]:
test_loss, test_pred, test_true, test_uuid, test_board = run_epoch(model, test_loader, device, optimizer=None)
overall_metrics = regression_metrics(test_true, test_pred)

pred_df = pd.DataFrame({
    "uuid": test_uuid,
    "board_key": test_board,
    "y_true": test_true,
    "y_pred": test_pred,
})
board_metrics_df = metrics_by_board(pred_df)

print("=" * 50)
print("Overall joint test performance")
print("=" * 50)
for key, value in overall_metrics.items():
    suffix = "%" if "within" in key or "exact" in key else ""
    print(f"{key:24s}: {value:8.4f}{suffix}")

print("\nBoard-specific test performance:")
print(board_metrics_df.to_string(index=False))

## Save Model and Artifacts

We save the trained model checkpoint and evaluation metrics for use in notebook 04 (route evaluation) and for future inference.

In [ ]:
# Save model checkpoint
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = ROOT / "data" / "processed" / "grade_prediction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the full model checkpoint (needed by notebook 04)
checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": len(stoi),
        "max_len": max_len,
        "d_model": 128,
        "nhead": 4,
        "num_layers": 4,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pad_id": pad_id,
    },
    "stoi": stoi,
    "itos": {str(k): v for k, v in itos.items()},
    "coord_features": coord_features.cpu(),
    "overall_metrics": overall_metrics,
}
model_path = MODEL_DIR / "joint_transformer_grade_predictor.pth"
torch.save(checkpoint, model_path)

# Save training history and metrics
pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)
board_metrics_df.to_csv(OUT_DIR / "board_metrics.csv", index=False)

from climbingboardgpt.utils import write_json
write_json(OUT_DIR / "overall_metrics.json", overall_metrics)

print(f"Saved model checkpoint to: {model_path}")
print(f"Saved training history to: {OUT_DIR / 'training_history.csv'}")
print(f"Saved test predictions to: {OUT_DIR / 'test_predictions.csv'}")
print(f"Saved board metrics to: {OUT_DIR / 'board_metrics.csv'}")

## Key Takeaways

1. **The transformer can learn from raw token sequences** without hand-engineered features like "mean hand reach" or "height gained". The self-attention mechanism lets it discover these patterns.

2. **Coordinate features help**: Injecting physical (x, y) position information gives the model a strong prior about spatial relationships, similar to how positional embeddings help language models.

3. **Joint training across boards**: By training on both TB2 and Kilter data simultaneously, the model can share statistical strength. The board token (`<BOARD_TB2>` vs `<BOARD_KILTER>`) tells it which "language" it's operating in.

4. **The gap between fine-grained and grouped metrics**: Being off by 1 difficulty point often stays within the same V-grade bucket. This is why the ±1 V-grade accuracy is much higher than the ±1 difficulty accuracy.